# Lesson 26: Probability calibration, Brier score, and log loss

ROC-AUC measures how well a model ranks observations. It does not tell us whether its predicted probabilities are accurate

Suppose a model predicts a fraud probability of 0.8. We want approximately 80% of transactions receiving predictions near 0.8 to actually be fraudulent

This property is called **calibration**


## Ranking versus calibration

Imagine two models produce these scores:

| Actual Y | Model A | Model B |
| --- | --- | --- |
| 1 | 0.90 | 0.60 |
| 0 | 0.80 | 0.55 |
| 1 | 0.70 | 0.52 |
| 0 | 0.10 | 0.45 |

Both models rank the observations in exctly the same order. Consequently, they have the same ROC-AUC.

However, their probabilities are different. Metrics such as Brier score and log loss can measure this difference


## Brier score

For binary classification:

$$ Brier = \frac{1}{n} \sum_{i=1}^{n} (p_i - y_i)^2 $$

Here:

  - $p_i$ is the predicted probability
  - $y_i$ is the actual result, either 0 or 1

Lower values are better. A perfect model has:

$$ Brier = 0 $$

### Example:

Suppose:

$$ y = 1, \quad p = 0.8 $$

The squared error is:

$$ (0.8 - 1)^2 = 0.04 $$

If the prediction were $p = 0.2$:

$$ (0.2 - 1)^2 = 0.64 $$

The second prediction receives a much larger penalty


## Log loss

Log loss is:

$$ LogLoss = - \frac{1}{n} \sum_{i=1}^n [y_i ln(p_i) + (1 - y_i) ln(1 - p_i)] $$

Lower values are better

For a positive observation, y = 1, the formula becomes:

$$ Loss = -ln(p) $$

If $p = 0.9$:

$$ Loss = -ln(0.9) \approx 0.1053 $$

If $p = 0.1$:

$$ Loss = -ln(0.1) \approx 2.3026 $$

Log loss strongly penalizes confident incorrect predictions

For a negative observation, $y = 0$, the formula becomes:

$$ Loss = -ln(1 - p) $$

Thus, predicting $p=0.99$ when $y = 0$ receives a very large penalty


## Brier score versus log loss

| Metric | Important property |
| :--- | :--- |
| Brier score | Uses squared probability errors |
| Log loss | Strong penalizes confident mistakes |
| ROC-AUC | Measures ranking rather than probability accuracy |

A model can gave a high ROC-AUC but poor Brier score and log loss



## Calibration curve

To create a calibration curve:

  1. divide predictions into probability groups
  2. calculate the average predicted probability in each group
  3. calculate the actual positive rate in each group
  4. compare the two values

For example:

| Average prediction | Actual fraud rate |
| --- | --- |
| 0.10 | 0.08 |
| 0.40 | 0.50 |
| 0.80 | 0.60 |


A perfectly calibrated model would produce points close to:

$$ Actual~rate = Predicted~probabillity $$


### Overconfidence

If the model predicts 0.80, but only 60% are fraudulent, it is overconfident:

$$ 0.80 > 0.60 $$

### Underconfidence

If the model predicts 0.4 but 50% are fraudulent, it is underconfident:

$$ 0.40 < 0.50 $$


## Probability calibration methods

Two common methods are:

  - Platt scaling: fits a logistic transformation
  - isotonic regression: fits a more flexible monotonic transformation

Calibration must be fitted using separate validation or calibration data. we must not fit it on the final test set


## Why calibration matters for business costs

We previously used:

$$ t = \frac{C_{FP}}{C_{FP} + C_{FN}} $$

This formula assumes that $p$ is a meaningful probability

If the model outputs $p = 0.20$, but the real fraud rate among such predictions is only 2%, a cost-based threshold from that probability may produce poor decisions

Therefore:

  - ROC-AUC helps evaluate ranking
  - calibration metrics evaluate probabilities
  - business costs help select the final decision threshold

## Practice

### Task 1

Use these predictions:

| Actual Y | Model A | Model B |
| ---: | ---: | ---: |
| 1 | 0.90 | 0.60 |
| 0 | 0.80 | 0.55 |
| 1 | 0.70 | 0.52 |
| 0 | 0.10 | 0.45 |

1. Explain why the models have the same ROC-AUC
2. Calculate the Brier score for Model A
3. Calculate the Brier score for Model B
4. Which model has the better Brier score?
5. Can models with the same ROC-AUC have different probability quality?


**Solution:**

1. Because ROC-AUC evalulates ranking across thresholds

2. Brier score for Model A

$$ Brier_{A} = \frac{(0.9 - 1)^2 + (0.8 - 0)^2 + (0.7 - 1)^2 + (0.1 - 0)^2}{4} = \frac{0.01 + 0.64 + 0.09 + 0.01}{4} = 0.1875 $$

3. Brier score for Model B

$$ Brier_{B} = \frac{(0.6 - 1)^2 + (0.55 - 0)^2 + (0.52 - 1)^2 + (0.45 - 1)^2}{4} = \frac{0.16 + 0.3025 + 0.2304 + 0.2025}{4} = 0.22385 $$

4. Model A has better Brier score

5. Yes, they can because they can be diffirent in calibration


### Task 2

Calculate the individual log loss for:

1. $y = 1, \ p = 0.8$
2. $y = 1, \ p = 0.2$
3. $y = 0, \ p = 0.8$
4. $y = 0, \ p = 0.2$

Use:

$$ ln(0.8) \approx -0.2231, \quad ln(0.2) \approx -1.6094 $$

Explain why confident incorrect predictions receive a large penalty


**Solution:**

$$ LogLoss_{1} = -ln(0.8) = -0.2231 $$

$$ Logoss_{2} = -ln(0.2) = -1.6094 $$

$$ LogLoss_{3} = -ln(1 - 0.8) = -1,6094 $$

$$ LogLoss_{4} = -ln(1 - 0.2) = -0.2231 $$


Confident incorrect predictions recieve a large penalty in LogLoss because the large difference between predicted probability and actual value yields a smaller logarithm and small log-value result in large error values, and on the contrary large log-values result in small error values


### Task 3

A calibration analysis produces:

| Predicted probability | Transactions | Actual frauds |
| ---: | ---: | ---: |
| 0.10 | 1000 | 80 |
| 0.40 | 500 | 250 |
| 0.80 | 100 | 60 |

For each group:

  1. calculate the actual fraud rate
  2. determine whether the model is calibrated, overconfident, or underconfident
  3. explain the meaning of yuor result


**Solution:**

1. Fraud rate

$$ Fraud~rate_{1} = \frac{80}{1000} = 8\% $$

$$ Fraud~rate_{2} = \frac{250}{500} = 50\% $$

$$ Fraud~rate_{3} = \frac{60}{100} = 60\% $$

2.

  * 1 group - slightly overconfident
  * 2 group - underconfident
  * 3 group - overconfident

3. From this three models first has the best calibration, but it also slightly overconfident, it means that in reality probability of fraud in data is 10%, but model predicts only 8%. Second model on the contrary underconfident, it means that model predicts 50% fraud, but in reality there is only 40%. Third model like first is overconfident


## Task 4

1. Can two models have the same ROC-AUC but different Brier scores?

  - Yes, they can, and this is normal situation

2. Does a high ROC-AUC guarantee calibrated probabilities?

  - No, it doesn't. For calibration evaluation specific metrics like logloss and brier score are used

3. Why does cost-based threshold selection require good calibration?

  - Because model can be overconfident or underconfident in predictions, this leads to mistakes in costs estimating, so business can lose some money because of these errors

4. Should probability calibration be fitted on the final test set?

  - No, final test can't be used for model fitting

5. Does a lower Brier score always guarantee a lower business cost at one particular threshold?

  - No, it doesn't. Business cost also depends on ranking, the selected threshold, class prevalence, and the relative costs of false positives and false negatives

